In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import os

# Configuración de dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


In [4]:
# --- 1. Definición de la Arquitectura (Ajustada a tu .pth) ---
class SimpleCNN2Digits(nn.Module):
    def __init__(self):
        super(SimpleCNN2Digits, self).__init__()
        # Arquitectura de 16 y 32 filtros (la que encaja con tu archivo)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Entrada: 32 canales * 7 alto * 14 ancho = 3136
        self.fc1 = nn.Linear(32 * 7 * 14, 128)
        self.fc2 = nn.Linear(128, 100) # 100 clases (00-99)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 14) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- 2. Carga del Modelo ---
model_path = "../models/simple_cnn_2digits.pth"
model = SimpleCNN2Digits().to(device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("✅ Modelo cargado y listo para inferencia.")
else:
    print(f"❌ Error: No se encontró el modelo en {model_path}")

# --- 3. Carga de los Datasets (Las tres piezas) ---
data_dir = "../data/raw_input"

try:
    # A. El dataset tabular (Ligero)
    df_tabular = pd.read_csv(os.path.join(data_dir, "m_tabular.csv"))
    
    # B. El dataset de imágenes de clientes (Pickle)
    df_cust_img = pd.read_pickle(os.path.join(data_dir, "m_cust_images.pkl"))
    
    # C. El dataset de imágenes de vendedores (Pickle)
    df_sell_img = pd.read_pickle(os.path.join(data_dir, "m_sell_images.pkl"))
    
    print("\n✅ Datasets cargados exitosamente:")
    print(f"- Tabular: {df_tabular.shape}")
    print(f"- Imágenes Cliente: {df_cust_img.shape}")
    print(f"- Imágenes Vendedor: {df_sell_img.shape}")

except FileNotFoundError as e:
    print(f"❌ Error al cargar los datos: {e}")

# Verificación rápida del contenido
display(df_cust_img.head(2))

✅ Modelo cargado y listo para inferencia.

✅ Datasets cargados exitosamente:
- Tabular: (119143, 40)
- Imágenes Cliente: (119143, 2)
- Imágenes Vendedor: (119143, 2)


,order_id,customer_state_image
0,e481f51cbdc54678b7cc49136f2d6af7,"[[[tensor(0.), tensor(0.), tensor(0.), tensor(..."
1,e481f51cbdc54678b7cc49136f2d6af7,"[[[tensor(0.), tensor(0.), tensor(0.), tensor(..."


In [5]:
# --- CELDA 1: Inferencia Clientes ---
print("🔍 Procesando imágenes de Clientes...")
model.eval()
customer_preds = []

with torch.no_grad():
    for img in df_cust_img['customer_state_image']:
        # Solo procesamos si es un Tensor de PyTorch (evitamos None, NaN o -1)
        if isinstance(img, torch.Tensor):
            input_tensor = img.unsqueeze(0).to(device)
            output = model(input_tensor)
            pred = torch.argmax(output, dim=1).item()
            customer_preds.append(pred)
        else:
            # Si no hay imagen, devolvemos -1 por regla de negocio
            customer_preds.append(-1)

# Guardamos el resultado y liberamos memoria eliminando los tensores
df_cust_img['customer_state_num_pred'] = customer_preds
df_cust_img.drop(columns=['customer_state_image'], inplace=True)

print(f"✅ Inferencia de clientes finalizada. IDs recuperados: {len(customer_preds)}")
display(df_cust_img.head(3))

🔍 Procesando imágenes de Clientes...
✅ Inferencia de clientes finalizada. IDs recuperados: 119143


,order_id,customer_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25
1,e481f51cbdc54678b7cc49136f2d6af7,25
2,e481f51cbdc54678b7cc49136f2d6af7,25


In [6]:
# --- CELDA 2: Inferencia Vendedores ---
print("🔍 Procesando imágenes de Vendedores...")
seller_preds = []

with torch.no_grad():
    for img in df_sell_img['seller_state_image']:
        # Verificación estricta de objeto Tensor
        if isinstance(img, torch.Tensor):
            input_tensor = img.unsqueeze(0).to(device)
            output = model(input_tensor)
            pred = torch.argmax(output, dim=1).item()
            seller_preds.append(pred)
        else:
            # Consistencia con la regla: Sin imagen = -1
            seller_preds.append(-1)

# Guardamos el resultado y limpiamos la columna pesada
df_sell_img['seller_state_num_pred'] = seller_preds
df_sell_img.drop(columns=['seller_state_image'], inplace=True)

print(f"✅ Inferencia de vendedores finalizada. IDs recuperados: {len(seller_preds)}")
display(df_sell_img.head(3))

🔍 Procesando imágenes de Vendedores...
✅ Inferencia de vendedores finalizada. IDs recuperados: 119143


,order_id,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25
1,e481f51cbdc54678b7cc49136f2d6af7,25
2,e481f51cbdc54678b7cc49136f2d6af7,25


In [15]:
# --- CELDA 3: Inyección de columnas (Estilo BUSCARV / Mapeo) ---
print("💉 Inyectando predicciones en la tabla principal...")

# 1. Convertimos las tablas de imágenes en "Diccionarios" de Pandas (Series)
# Nos aseguramos de quitar duplicados para que a cada 'order_id' le corresponda solo un estado
dict_cliente = df_cust_img.drop_duplicates('order_id').set_index('order_id')['customer_state_num_pred']
dict_vendedor = df_sell_img.drop_duplicates('order_id').set_index('order_id')['seller_state_num_pred']

# 2. Añadimos la información como columnas nuevas directamente a df_tabular
# Usamos .map() para buscar el order_id en nuestros "diccionarios"
df_tabular['customer_state_num_pred'] = df_tabular['order_id'].map(dict_cliente)
df_tabular['seller_state_num_pred'] = df_tabular['order_id'].map(dict_vendedor)

# 3. Limpieza: Si algún order_id no tenía imagen, map() devuelve NaN. Lo rellenamos con -1.
df_tabular['customer_state_num_pred'] = df_tabular['customer_state_num_pred'].fillna(-1).astype(int)
df_tabular['seller_state_num_pred'] = df_tabular['seller_state_num_pred'].fillna(-1).astype(int)

# Ahora m_final es exactamente df_tabular pero con las dos columnas extra
m_final = df_tabular.copy()

print("✅ Columnas añadidas con éxito.")
print(f"Dimensiones finales del dataset: {m_final.shape}") # ¡Volverá a ser 119143, 42!

# Comprobación visual
display(m_final[['order_id', 'customer_state_num', 'customer_state_num_pred', 'seller_state_num_pred']].head())

💉 Inyectando predicciones en la tabla principal...
✅ Columnas añadidas con éxito.
Dimensiones finales del dataset: (119143, 42)


,order_id,customer_state_num,customer_state_num_pred,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
1,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
2,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
3,53cdb2fc8bc7dce0b6741e2150273451,4,4,25
4,47770eb9100c2d0c44946d9cf07ec65d,8,8,25


In [16]:
# Verificación de "Huecos" detectados
vacios_cliente = (df_cust_img['customer_state_num_pred'] == -1).sum()
vacios_vendedor = (df_sell_img['seller_state_num_pred'] == -1).sum()

print(f"Total de registros: {len(df_cust_img)}")
print(f"Filas donde la IA NO intervino (Clientes): {vacios_cliente}")
print(f"Filas donde la IA NO intervino (Vendedores): {vacios_vendedor}")

Total de registros: 119143
Filas donde la IA NO intervino (Clientes): 0
Filas donde la IA NO intervino (Vendedores): 833


In [17]:
import os

# Asegurarnos de que la carpeta existe (por si acaso)
os.makedirs('../data', exist_ok=True)

# Definir las rutas (he corregido el nombre a 'Integrated', cámbialo si el error era intencional)
ruta_pkl = '../data/Integrated.pkl'

# 2. Guardar como Pickle (Opcional, pero muy recomendado para no perder los tipos de datos 'int' de tus IDs)
m_final.to_pickle(ruta_pkl)
print(f"✅ Dataset maestro guardado en formato Pickle: {ruta_pkl}")

✅ Dataset maestro guardado en formato Pickle: ../data/Integrated.pkl
